# Positional Encoding
### Teaching a model that "knows everything at once" to understand "before" and "after"

---

## 📖 The Story

Self-attention (Topic 2) and multi-head attention (Topic 3) let every token interact with every other token in one parallel operation — but nothing in that math distinguishes "first" from "last." Feed the model "The dog bit the man" and "The man bit the dog," and self-attention alone produces the exact same attention weights, just reassigned to different tokens.

The Transformer's creators ask: *"How do we tell the model where each token sits, without bringing back the RNN's slow sequential processing?"* Their answer: stamp a fixed sine/cosine pattern directly onto each token's embedding BEFORE attention ever runs. This notebook builds that exact mechanism: **positional encoding**.

---

**What this notebook covers:**
- Demonstrating the permutation invariance problem directly with our own self-attention code
- Building the sine/cosine positional encoding matrix from scratch with NumPy
- Visualizing the wave patterns across positions and dimensions
- Adding positional encoding to embeddings and confirming the order-blindness problem is fixed
- Verifying our implementation against PyTorch

**Prerequisites:**
- Why Attention Matters (Topic 1)
- Self-Attention (Topic 2)
- Multi-Head Attention (Topic 3)

**Note:** Like Topics 1–3, this notebook uses an in-notebook example sentence rather than an external dataset — positional encoding is best understood through direct, inspectable computation on a small example.

In [ ]:
# --- All Imports ---
import numpy as np                        # Core math — positional encoding from scratch
import matplotlib.pyplot as plt           # Plotting
import seaborn as sns                     # Heatmap visualizations
import torch                              # PyTorch — verification
import torch.nn as nn                     # nn.Module for our PE layer
import math
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)                        # Reproducibility
torch.manual_seed(42)

print("All imports successful ✅")

## Part 1: Theory Recap

Five things to know before we write any code:

- **Self-attention is permutation invariant**: Topics 2 and 3 showed attention computed purely from Query-Key dot products of embeddings — nothing in that formula references WHERE a token sits, only WHAT it contains.
- **Positional encoding injects position before attention runs**: A fixed pattern, unique to each position, is added directly to the token embedding — so the input to attention already carries order information.
- **The formula uses sine and cosine waves of different frequencies**: PE(pos,2i) = sin(pos/10000^(2i/d_model)), PE(pos,2i+1) = cos(pos/10000^(2i/d_model)) — low dimensions oscillate fast, high dimensions oscillate slowly.
- **Values stay bounded between -1 and 1**: Unlike raw integer positions (0, 1, 2, ...), sine and cosine never grow unbounded, so positional encoding does not distort embeddings even for long sequences.
- **No extra learned parameters (sinusoidal version)**: The encoding is precomputed from a fixed formula — it costs nothing extra to train and can be evaluated at positions never seen during training.

## Setting Up a Real Example Sentence

We use the sentence **"The cat sat on mat"** — the same sentence from Topics 1–3 — so you can directly see how positional encoding changes the input these earlier mechanisms receive.

We will also build a SECOND, shuffled version of the same words, to concretely demonstrate the permutation invariance problem before we fix it.

In [ ]:
# Define our example sentence
words = ["The", "cat", "sat", "on", "mat"]
seq_len   = len(words)
d_model   = 8   # embedding dimension — same as Topics 2 and 3

# A shuffled version of the SAME words, used later to demonstrate permutation invariance
shuffled_order = [2, 0, 4, 1, 3]   # indices into `words`
shuffled_words = [words[i] for i in shuffled_order]

# Create input embeddings — one vector per word
# In a real model these come from a trained embedding layer
X = np.random.randn(seq_len, d_model) * 0.5

print("Input embeddings X:")
print(f"Shape: {X.shape}  (seq_len={seq_len}, d_model={d_model})\n")
for word, vec in zip(words, X):
    print(f"  {word:6s}: {np.round(vec, 2)}")

print(f"\nOriginal order : {words}")
print(f"Shuffled order : {shuffled_words}")

## Part 2: Demonstrating the Permutation Invariance Problem

Before building the fix, let's PROVE the problem exists using our own self-attention code from Topic 2.

We will run self-attention on the original sentence and on the shuffled sentence (same words, same embeddings, just reordered), and show that the attention OUTPUT for the shuffled version is simply the same values in a different order — not a genuinely different computation.

In [ ]:
def softmax(x, axis=-1):
    """
    Numerically stable softmax along a given axis.
    """
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


def self_attention_no_position(X, d_k):
    """
    Plain self-attention (Topic 2), with NO positional encoding.
    Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V
    Uses a SHARED random projection per call so results are comparable.
    """
    np.random.seed(0)   # same weights every call — isolates the effect of ORDER, not random weights
    W_Q = np.random.randn(X.shape[1], d_k) * 0.5
    W_K = np.random.randn(X.shape[1], d_k) * 0.5
    W_V = np.random.randn(X.shape[1], d_k) * 0.5

    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    scores = (Q @ K.T) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights


d_k = d_model

# Run on the ORIGINAL order
output_original, weights_original = self_attention_no_position(X, d_k)

# Run on the SHUFFLED order — using the SAME embeddings, just reordered rows
X_shuffled = X[shuffled_order]
output_shuffled, weights_shuffled = self_attention_no_position(X_shuffled, d_k)

# To compare fairly, "unshuffle" the shuffled output back to original word order
unshuffle_index = np.argsort(shuffled_order)
output_shuffled_unsorted = output_shuffled[unshuffle_index]

print("Output for 'sat' (word at original position 2):")
print(f"  From ORIGINAL order run : {np.round(output_original[2], 5)}")
print(f"  From SHUFFLED order run : {np.round(output_shuffled_unsorted[2], 5)}")

max_diff = np.max(np.abs(output_original - output_shuffled_unsorted))
print(f"\nMax absolute difference after unshuffling: {max_diff:.2e}")
print("⚠️  IDENTICAL (up to floating point) — self-attention literally cannot tell the two")
print("    word orders apart. This is the permutation invariance problem positional encoding fixes.")

## What Just Happened?

We just proved, with real numbers, that plain self-attention is blind to word order.

- We ran self-attention on "The cat sat on mat" and got one set of outputs
- We ran self-attention on a SHUFFLED version of the same words — same embeddings, same weights, just reordered
- After "unshuffling" the second run's output back into original word order, the two outputs are numerically IDENTICAL
- This confirms: self-attention's output for a given word depends only on WHICH other words are present, never on the ORDER they appear in

This is exactly the problem positional encoding exists to solve. Let's build the fix.

## Part 3: Positional Encoding From Scratch

We now build the sine/cosine positional encoding matrix step by step:
1. Compute the frequency-scaling term for each dimension pair
2. Fill even dimensions with sine values, odd dimensions with cosine values
3. Add the resulting PE matrix directly to the token embeddings
4. Re-run self-attention and confirm the shuffling problem is now fixed

In [ ]:
def positional_encoding(seq_len, d_model):
    """
    Sinusoidal Positional Encoding (Vaswani et al., 2017).
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

    Args:
        seq_len : number of positions to encode
        d_model : embedding dimension

    Returns:
        pe : positional encoding matrix — shape (seq_len, d_model)
    """
    pe = np.zeros((seq_len, d_model))
    positions = np.arange(seq_len)[:, np.newaxis]              # shape (seq_len, 1)
    dim_indices = np.arange(0, d_model, 2)                     # 0, 2, 4, ... — one per (sin, cos) pair

    # Step 1: Frequency-scaling term — controls how fast each dimension's wave oscillates
    div_term = np.power(10000.0, dim_indices / d_model)        # shape (d_model/2,)

    # Step 2: Fill even dimensions with sine, odd dimensions with cosine
    pe[:, 0::2] = np.sin(positions / div_term)
    pe[:, 1::2] = np.cos(positions / div_term)

    return pe


PE = positional_encoding(seq_len, d_model)

print(f"Positional encoding matrix shape: {PE.shape}  (seq_len={seq_len}, d_model={d_model})\n")
for word, pos, pe_vec in zip(words, range(seq_len), PE):
    print(f"  pos={pos} ({word:4s}): {np.round(pe_vec, 3)}")

print("\nNotice every position gets a UNIQUE pattern, and all values stay between -1 and 1.")

In [ ]:
# Step 3: Add positional encoding directly to the token embeddings
X_with_pe = X + PE

print("Embedding for 'sat' BEFORE adding positional encoding:")
print(f"  {np.round(X[2], 3)}")
print("Embedding for 'sat' AFTER adding positional encoding:")
print(f"  {np.round(X_with_pe[2], 3)}")
print("\n(Same word, now shifted by a position-specific signature)\n")

# Step 4: Re-run the SAME shuffling experiment from Part 2, but now WITH positional encoding
# IMPORTANT (the realistic scenario): when words are physically reordered, each word now SITS
# in a different SLOT — so it must receive the PE for its NEW slot, not carry its old PE along.
# We therefore shuffle the RAW embeddings first, then add PE based on the new position.
X_shuffled_raw = X[shuffled_order]        # words physically reordered (no PE yet)
X_shuffled_with_pe = X_shuffled_raw + PE  # each word now gets the PE for its NEW slot

output_pe_original, _ = self_attention_no_position(X_with_pe, d_k)
output_pe_shuffled, _ = self_attention_no_position(X_shuffled_with_pe, d_k)
output_pe_shuffled_unsorted = output_pe_shuffled[unshuffle_index]

print("Output for 'sat' WITH positional encoding:")
print(f"  From ORIGINAL order run : {np.round(output_pe_original[2], 5)}")
print(f"  From SHUFFLED order run : {np.round(output_pe_shuffled_unsorted[2], 5)}")

max_diff_pe = np.max(np.abs(output_pe_original - output_pe_shuffled_unsorted))
print(f"\nMax absolute difference: {max_diff_pe:.2e}")
print("✅ Now DIFFERENT — because 'sat' now receives the positional signature for whichever")
print("    SLOT it occupies, and self-attention's output changes accordingly. Order now matters.")


In [ ]:
# --- Visualisation 1: The Positional Encoding Wave Pattern ---
# This is the classic "stripes" visualization seen in every transformer blog post —
# each row is a position, each column is a dimension, color shows the sin/cos value.

# Use a longer sequence here so the wave pattern is clearly visible
demo_seq_len = 50
demo_d_model = 64
PE_demo = positional_encoding(demo_seq_len, demo_d_model)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(PE_demo, cmap='RdBu', aspect='auto')
ax.set_title('Positional Encoding Matrix — Sine/Cosine Waves Across Positions and Dimensions',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Embedding Dimension', fontsize=11)
ax.set_ylabel('Position in Sequence', fontsize=11)
fig.colorbar(im, ax=ax, label='Encoding Value')
plt.tight_layout()
plt.show()

print("Notice the STRIPES: columns on the left (low dimension index) oscillate quickly —")
print("changing every position or two. Columns on the right (high dimension index) oscillate")
print("slowly — changing gradually across many positions. Together, every ROW (position) gets")
print("a unique combination of fast and slow wave values — a unique fingerprint.")

In [ ]:
# --- Visualisation 2: Individual Wave Curves at Different Frequencies ---
# Shows WHY different dimensions oscillate at different speeds — directly plotting
# a few individual sin/cos curves as functions of position.

positions_demo = np.arange(0, 100)
dims_to_show = [0, 2, 8, 32]   # a spread of low to high dimension indices

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for dim in dims_to_show:
    div = np.power(10000.0, dim / demo_d_model)
    wave = np.sin(positions_demo / div)
    axes[0].plot(positions_demo, wave, label=f'dim={dim}', linewidth=2)

axes[0].set_title('Sine Component at Different Dimension Indices', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Position', fontsize=11)
axes[0].set_ylabel('PE value', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right plot: how wavelength grows with dimension index
dim_range = np.arange(0, demo_d_model, 2)
wavelengths = 2 * np.pi * np.power(10000.0, dim_range / demo_d_model)
axes[1].plot(dim_range, wavelengths, marker='o', color='darkorange', linewidth=2, markersize=5)
axes[1].set_title('Wavelength Grows With Dimension Index\n(low dims = fast waves, high dims = slow waves)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Dimension Index (2i)', fontsize=11)
axes[1].set_ylabel('Wavelength (positions per full cycle)', fontsize=11)
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Why Positional Encoding Uses Multiple Frequencies', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Low dimension indices complete a full sine wave cycle in just a few positions —")
print("good for distinguishing NEARBY positions. High dimension indices take thousands of")
print("positions to complete one cycle — good for distinguishing FAR-APART positions.")
print("Together, this multi-frequency design gives every position a distinguishable signature.")

## Part 4: PyTorch Verification

We verify our scratch implementation using a small PyTorch `nn.Module` that follows the same formula as the original "Attention Is All You Need" paper. This is functionally identical to the positional encoding layer used in real Transformer implementations.

In [ ]:
class PositionalEncodingTorch(nn.Module):
    """
    PyTorch implementation of sinusoidal positional encoding,
    following Vaswani et al., 2017 exactly.
    """
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, seq_len):
        return self.pe[:seq_len]


pe_torch_layer = PositionalEncodingTorch(d_model=d_model, max_len=100)
PE_torch = pe_torch_layer(seq_len).numpy()

print("=== Scratch vs PyTorch Verification ===\n")
print("Positional encoding for position 2 ('sat'):")
print(f"  Scratch : {np.round(PE[2], 5)}")
print(f"  PyTorch : {np.round(PE_torch[2], 5)}")

max_diff = np.max(np.abs(PE - PE_torch))
print(f"\nMax absolute difference across full PE matrix: {max_diff:.2e}")
print("✅ Match — our scratch implementation matches the standard PyTorch formulation" if max_diff < 1e-4
      else "❌ Mismatch — check the frequency formula or sin/cos indexing")

## Part 5: Hyperparameter Experiments

Two experiments to build deeper intuition:

1. **Effect of d_model on how distinguishable nearby positions are** — does a larger embedding dimension make positions easier to tell apart?
2. **Generalization beyond training length** — directly demonstrating that the sinusoidal formula can be evaluated at ANY position, even ones far beyond what a model might have been trained on.

In [ ]:
# Experiment 1: Does increasing d_model make adjacent positions more distinguishable?
d_model_values = [4, 8, 16, 32, 64, 128]
adjacent_distances = []

for d in d_model_values:
    pe_test = positional_encoding(seq_len=2, d_model=d)   # just positions 0 and 1
    dist = np.linalg.norm(pe_test[0] - pe_test[1])         # Euclidean distance between PE(0) and PE(1)
    adjacent_distances.append(dist)

# Experiment 2: Generalization — evaluate PE at positions FAR beyond a typical training length
train_max_len = 50
test_positions = [10, 50, 100, 500, 2000, 10000]   # some well beyond train_max_len
pe_far = positional_encoding(seq_len=max(test_positions) + 1, d_model=64)
sample_vectors = [pe_far[p] for p in test_positions]
# Confirm values are still bounded, even at position 10,000
value_ranges = [(v.min(), v.max()) for v in sample_vectors]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(d_model_values, adjacent_distances, marker='o', color='steelblue', linewidth=2, markersize=8)
axes[0].set_title('Distance Between PE(0) and PE(1) vs d_model', fontsize=12, fontweight='bold')
axes[0].set_xlabel('d_model', fontsize=11)
axes[0].set_ylabel('Euclidean Distance', fontsize=11)
axes[0].grid(True, alpha=0.3)

mins = [r[0] for r in value_ranges]
maxs = [r[1] for r in value_ranges]
x_pos = np.arange(len(test_positions))
axes[1].bar(x_pos - 0.2, mins, width=0.4, label='min value', color='salmon')
axes[1].bar(x_pos + 0.2, maxs, width=0.4, label='max value', color='seagreen')
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(-1.0, color='gray', linestyle='--', linewidth=1)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(test_positions)
axes[1].set_title('PE Values Stay Bounded [-1, 1] Even at Position 10,000', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Position', fontsize=11)
axes[1].set_ylabel('Value', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Positional Encoding Experiments: Dimension Effect & Unbounded Generalization',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Even at position 10,000 — 200x beyond a training length of {train_max_len} — every")
print("positional encoding value stays safely between -1 and 1. This is only possible because")
print("the formula is a closed-form sine/cosine function, not a lookup table with a fixed size.")

## Common Mistakes Students Make

**Mistake 1: Thinking positional encoding is learned (in the original Transformer)**
The sinusoidal version from Vaswani et al. is a FIXED, precomputed pattern — there is nothing to learn, and no gradient ever flows into it. This is different from BERT/GPT-style learned positional embeddings, which ARE trainable parameters. Mixing these two up is one of the most common interview mistakes.

**Mistake 2: Concatenating instead of adding**
Positional encoding is ADDED elementwise to the token embedding (X_embedding + PE), not concatenated as extra dimensions. Concatenating would change d_model and require redesigning every downstream weight matrix — addition keeps the shape identical and lets position blend smoothly into the existing embedding space.

**Mistake 3: Forgetting that PE must be added BEFORE attention, not after**
Positional encoding only works if it is added to the input embeddings before they are projected into Q, K, V. Adding it after self-attention has already computed its output is too late — the attention weights themselves would already have been computed with no order information.

## Part 6: Interview Corner

**Question: Walk me through why a fully attention-based model needs positional encoding, when an RNN does not.**

This directly addresses why Topics 2 and 3's powerful, fully parallel mechanisms still needed one more piece. Complete answer:

**Setup:** An RNN processes tokens one at a time, in order — the hidden state at step t is mathematically built FROM the hidden state at step t-1. Order is baked into the computation itself; there is no way to run an RNN "out of order."

**Step 1 — The gap:** Self-attention and multi-head attention compute every token's relationship to every other token in one parallel matrix operation. The Query-Key-Value formulas reference embeddings only — nothing in the math depends on which index a token came from.

**Step 2 — The consequence:** This makes attention permutation invariant. As demonstrated in Part 2 of this notebook, shuffling the input and un-shuffling the output produces numerically identical results to never shuffling at all.

**Step 3 — The fix:** Positional encoding adds a fixed, position-dependent signal to each token's embedding before any attention computation happens. Because the signal is unique per position, two identical words at different positions now produce different embeddings — and therefore different attention scores.

**Step 4 — Why sine/cosine specifically:** The wave-based formulation keeps values bounded regardless of sequence length, and gives nearby positions similar (but distinguishable) signatures while giving far-apart positions very different ones — letting attention implicitly learn distance relationships.

**The key insight:** Positional encoding is the one piece of "engineering" that lets a fully parallel, RNN-free architecture still respect sequence order — without sacrificing any of the parallelism gains from Topics 2 and 3. It is added once, costs nothing extra to compute (in its sinusoidal form), and then every subsequent attention layer automatically inherits order-awareness for free.

## Key Takeaways

Five bullets you must remember for placement interviews:

- **Self-attention alone is permutation invariant**: Topics 2 and 3's formulas reference only embeddings, never position — shuffling input tokens produces an identically shuffled output, with no order information preserved.
- **Positional encoding adds a fixed, position-dependent signal before attention runs**: PE(pos,2i) = sin(pos/10000^(2i/d_model)), PE(pos,2i+1) = cos(pos/10000^(2i/d_model)), added directly to the token embedding.
- **Different dimensions oscillate at different frequencies**: Low dimension indices change quickly (good for nearby positions), high dimension indices change slowly (good for far-apart positions) — together forming a unique fingerprint per position.
- **The sinusoidal version has zero extra trainable parameters and generalizes to unseen lengths**: Because it is a closed-form formula, it can be evaluated at any position, even ones beyond the longest sequence seen during training — unlike learned positional embeddings.
- **Positional encoding must be added BEFORE Q, K, V projections**: It needs to be part of the input that attention actually sees — adding it after attention has already run would be too late to have any effect on the attention weights.